# ⚡ Antigravity 4K GPU Multi-Worker Parallel Video Studio
Studio Produksi Massal Video Loop 4K (3840x2160, 30 FPS, Seamless Loop) dengan **Parallel GPU Multi-Worker Engine**.

### 🚀 Keunggulan Versi Paralel:
1. **Parallel Multi-Worker (3x - 4x Lebih Cepat)**: GPU Nvidia T4 (15 GB VRAM) bekerja memproses beberapa video 4K sekaligus secara simultan.
2. **Direct Fast Auto-Download**: Menggunakan Web Socket / Stream Blob langsung tanpa antrean lama browser.
3. **Ukuran File Presisi**: Setiap video 10 detik tetap padat di rentang **25 MB - 50 MB** standar Adobe Stock & Freepik.

## ⚙️ Step 1: Install Environment & Multi-Worker GPU Dependencies (Hanya 1x Klik)

In [ ]:
import os, subprocess, shutil
os.chdir('/content')

print("⏳ [1/3] Memasang Google Chrome & Library GPU Hardware...")
!wget -q -O /tmp/chrome.deb https://dl.google.com/linux/direct/google-chrome-stable_current_amd64.deb
!dpkg -i /tmp/chrome.deb > /dev/null 2>&1 || apt-get install -fy > /dev/null 2>&1
!apt-get install -y ffmpeg libgbm-dev libnss3 libasound2 zip > /dev/null 2>&1
!rm -f /tmp/chrome.deb

print("⏳ [2/3] Mengunduh skrip parallel renderer terbaru dari GitHub...")
if os.path.exists('/content/shadergradientpaper'):
    shutil.rmtree('/content/shadergradientpaper', ignore_errors=True)

!git clone https://github.com/consistmaker/shadergradientpaper.git /content/shadergradientpaper

print("⏳ [3/3] Menyiapkan modul WebGL & Puppeteer...")
%cd /content/shadergradientpaper
!npm install --legacy-peer-deps > /dev/null 2>&1
!npm install puppeteer-core > /dev/null 2>&1
!npm run build
%cd /content

print("\n✅ PARALLEL GPU ENGINE SIAP 100%! SILAKAN LANJUT KE STEP 2 DI BAWAH.")

## 📥 Step 2: Masukkan Resep JSON Antrean (Bisa 5 - 50 Video Sekaligus)
1. Buka Web Studio di browser Anda: 👉 **[https://shadergradientpaper.vercel.app](https://shadergradientpaper.vercel.app)**
2. Atur preset atau klik **+ Tambah ke Antrean Render** untuk beberapa video pilihan Anda.
3. Klik **Export Batch** -> **Copy JSON**.
4. Paste di bawah ini lalu jalankan Cell!

In [ ]:
import json
import os

# PASTE JSON RESEP DARI WEB DI SINI:
RECIPE_JSON = '''
{
  "metadata": {
    "targetResolution": "3840x2160 (4K UHD)",
    "targetFps": 30,
    "loopDurationSeconds": 10,
    "isSeamlessLoop": true,
    "batchMode": "manual_queue"
  },
  "manualQueueList": [
    {
      "index": 1,
      "id": "item_1",
      "name": "Paper: mesh-gradient (#ffffff)",
      "engine": "paper",
      "config": {
        "shaderType": "mesh-gradient",
        "color1": "#ffffff",
        "color2": "#000000",
        "color3": "#ffffff",
        "color4": "#000000",
        "speed": 1.0,
        "distortion": 1.0,
        "swirl": 0.2
      }
    },
    {
      "index": 2,
      "id": "item_2",
      "name": "Paper: mesh-gradient (#bcecf6)",
      "engine": "paper",
      "config": {
        "shaderType": "mesh-gradient",
        "color1": "#bcecf6",
        "color2": "#00aaff",
        "color3": "#00f7ff",
        "color4": "#ffd447",
        "speed": 0.1,
        "distortion": 0.8,
        "swirl": 0.35
      }
    }
  ],
  "totalVideosInQueue": 2
}
'''

with open('/content/render_recipe.json', 'w') as f:
    f.write(RECIPE_JSON.strip())

recipe = json.loads(RECIPE_JSON)
total_vids = len(recipe.get('manualQueueList', []))
print(f"🎯 Mode Render: {recipe['metadata'].get('batchMode', 'manual_queue').upper()}")
print(f"🎬 Total Antrean: {total_vids} Video 4K UHD @ {recipe['metadata'].get('targetFps', 30)} FPS")
print("✅ Resep JSON berhasil disimpan! Siap dieksekusi secara Paralel di Step 3.")

## 🚀 Step 3: Eksekusi GPU Multi-Worker Parallel Render & Fast Instant Download

In [ ]:
import subprocess, time, os, glob
from google.colab import files

output_dir = '/content/output_4k_videos'
os.makedirs(output_dir, exist_ok=True)
downloaded_files = set()

# Set jumlah video yang diproses bersamaan (Parallel GPU Workers)
# Nilai 3 - 4 adalah titik optimal untuk memaksimalkan 15GB VRAM Nvidia T4 Colab tanpa bottleneck
os.environ['CONCURRENCY'] = '3'

print("🚀 Starting Parallel GPU 4K Render Engine with Multi-Worker Pipeline...")

process = subprocess.Popen(
    ['node', '/content/shadergradientpaper/parallel_renderer.cjs'],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    universal_newlines=True,
    bufsize=1
)

while True:
    line = process.stdout.readline()
    if line:
        print(line, end='')
        if '✅ Success 4K Render:' in line:
            time.sleep(0.5)
            current_videos = glob.glob(f"{output_dir}/*.mp4")
            for v_path in current_videos:
                if v_path not in downloaded_files and os.path.exists(v_path):
                    file_size = (os.path.getsize(v_path) / (1024 * 1024))
                    print(f"   📥 [INSTANT AUTO-DOWNLOAD] Mengunduh: {os.path.basename(v_path)} ({file_size:.2f} MB)...")
                    try:
                        files.download(v_path)
                        downloaded_files.add(v_path)
                    except Exception as e:
                        pass

    if process.poll() is not None:
        for remaining in process.stdout.readlines():
            print(remaining, end='')
        break

# Pastikan tidak ada video yang tertinggal
all_final_videos = glob.glob(f"{output_dir}/*.mp4")
for v_path in all_final_videos:
    if v_path not in downloaded_files:
        print(f"📥 [DOWNLOAD FINAL]: {os.path.basename(v_path)}...")
        files.download(v_path)
        downloaded_files.add(v_path)

print(f"\n🎉 SELESAI SEMPURNA! Total {len(downloaded_files)} video 4K UHD telah diproses secara paralel dan terunduh ke laptop/PC Anda.")

## 📦 Step 4 (Opsional): Download Cadangan Seluruh Video Sebagai 1 File ZIP

In [ ]:
import os
from google.colab import files

if os.path.exists('/content/output_4k_videos') and len(os.listdir('/content/output_4k_videos')) > 0:
    !cd /content && zip -r /content/4K_Parallel_Batch_Videos.zip output_4k_videos
    print("\n📦 Packaging Complete! Mengunduh file backup ZIP...")
    files.download('/content/4K_Parallel_Batch_Videos.zip')
else:
    print("❌ Tidak ada video di folder output.")